In [5]:
import pandas as pd
import re
import emoji
import langdetect
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from datasets import Dataset

# Make sure necessary NLTK data is downloaded
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

# Load dataset
df = pd.read_csv('../data/cyberbullying_dataset_raw.csv')
# --- PREPROCESSING ---

# 1. Lowercasing
df['text'] = df['text'].str.lower()

# 2. Convert emojis to their text meaning
df['text'] = df['text'].apply(lambda x: emoji.demojize(x, language='en',delimiters=(' ',' ')))
def remove_consecutive_duplicates(text):
    tokens = text.split()
    cleaned_tokens = []
    for i, token in enumerate(tokens):
        if i == 0 or token != tokens[i-1]:
            cleaned_tokens.append(token)
    return ' '.join(cleaned_tokens)

df['text'] = df['text'].apply(remove_consecutive_duplicates)
# 3. Remove non-English tweets
def is_english(text):
    try:
        return langdetect.detect(text) == 'en'
    except:
        return False

df = df[df['text'].apply(is_english)]

# 4. Remove URLs and special characters
url_pattern = re.compile(r'https?://\S+|www\.\S+')
users_pattern = re.compile(r'@[a-zA-Z0-9_]+')
special_char_pattern = re.compile(r'[^a-zA-Z0-9\s]')
df['text'] = df['text'].apply(lambda x: users_pattern.sub('user', x))
df['text'] = df['text'].apply(lambda x: url_pattern.sub('', x))
df['text'] = df['text'].apply(lambda x: special_char_pattern.sub('', x))

# 5. Tokenize (whitespace)
df['text'] = df['text'].apply(lambda x: x.split())

# 6. Remove stopwords
stop_words = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda tokens: [t for t in tokens if t not in stop_words])

# 7. Lemmatization
lemmatizer = WordNetLemmatizer()
df['text'] = df['text'].apply(lambda tokens: [lemmatizer.lemmatize(t) for t in tokens])

# 8. Join tokens back into string
df['text'] = df['text'].apply(lambda tokens: ' '.join(tokens))

# Ensure exactly 6 distinct labels remain (optional check)
unique_labels = df['label'].nunique()
if unique_labels != 6:
    print(f"Warning: Found {unique_labels} unique labels, not 6.")

#drop rows with null text or empty text
df = df.dropna(subset=['text'])
df = df[df['text'] != ""]

# Save cleaned dataset to CSV if needed
df.to_csv('../data/cyberbullying_dataset.csv', index=False)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
